In [67]:
import pandas as pd
from functions import *

In [68]:
df = read_dataframe("war_economic_impact_dataset.csv")
df = format_columns(df)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 28 columns):
 #   Column                                   Non-Null Count   Dtype  
---  ------                                   --------------   -----  
 0   conflict_name                            100000 non-null  object 
 1   conflict_type                            100000 non-null  object 
 2   region                                   100000 non-null  object 
 3   start_year                               100000 non-null  int64  
 4   end_year                                 100000 non-null  int64  
 5   status                                   100000 non-null  object 
 6   primary_country                          100000 non-null  object 
 7   pre_war_unemployment_                    100000 non-null  float64
 8   during_war_unemployment_                 100000 non-null  float64
 9   unemployment_spike_percentage_points     100000 non-null  float64
 10  most_affected_sector             

In [69]:
#Delete columns we don't want 

df = df.drop(columns= ['status', 'pre_war_unemployment_',
   'during_war_unemployment_',                 
  'unemployment_spike_percentage_points',     
 'most_affected_sector',                     
 'youth_unemployment_change_',               
  'pre_war_poverty_rate_',                   
 'during_war_poverty_rate_',                 
 'extreme_poverty_rate_', 'households_fallen_into_poverty_estimate', 'inflation_rate_',                          
'currency_devaluation_',                    
'cost_of_war_usd',                          
'estimated_reconstruction_cost_usd',        
'informal_economy_size_pre_war_',           
'informal_economy_size_during_war_',        
'black_market_activity_level',              
'primary_black_market_goods',               
'currency_black_market_rate_gap_',          
'war_profiteering_documented', 'gdp_change_'])


In [70]:
df

,conflict_name,conflict_type,region,start_year,end_year,primary_country,food_insecurity_rate_
0,WWII (Japan),World War,East Asia,1939,1945,Japan,15.00
1,Israel-Hamas War,Asymmetric War,Middle East,2023,2026,Palestine (Gaza),12.92
2,Syrian Civil War,Civil War,Middle East,2011,2026,Syria,28.83
3,WWII (Germany),World War,Europe,1939,1945,Germany,5.40
4,Afghanistan War,Interstate/Counter-insurgency,South Asia,2001,2021,Afghanistan,12.33
...,...,...,...,...,...,...,...
99995,Afghanistan War,Interstate/Counter-insurgency,South Asia,2001,2021,Afghanistan,4.37
99996,Israel-Iran War,Interstate War,Middle East,2025,2026,Iran,8.71
99997,Israel-Iran War,Interstate War,Middle East,2025,2026,Iran,20.19
99998,Israel-Hamas War,Asymmetric War,Middle East,2023,2026,Palestine (Gaza),11.20


In [71]:
df = df[(df["start_year"] >= 1990)]

In [72]:
df

,conflict_name,conflict_type,region,start_year,end_year,primary_country,food_insecurity_rate_
1,Israel-Hamas War,Asymmetric War,Middle East,2023,2026,Palestine (Gaza),12.92
2,Syrian Civil War,Civil War,Middle East,2011,2026,Syria,28.83
4,Afghanistan War,Interstate/Counter-insurgency,South Asia,2001,2021,Afghanistan,12.33
7,Israel-Hamas War,Asymmetric War,Middle East,2023,2026,Palestine (Gaza),20.20
8,Afghanistan War,Interstate/Counter-insurgency,South Asia,2001,2021,Afghanistan,5.99
...,...,...,...,...,...,...,...
99995,Afghanistan War,Interstate/Counter-insurgency,South Asia,2001,2021,Afghanistan,4.37
99996,Israel-Iran War,Interstate War,Middle East,2025,2026,Iran,8.71
99997,Israel-Iran War,Interstate War,Middle East,2025,2026,Iran,20.19
99998,Israel-Hamas War,Asymmetric War,Middle East,2023,2026,Palestine (Gaza),11.20


In [73]:
#Delete complete duplicates

df.duplicated().sum()
df = df.drop_duplicates()

#Check for null values
df = df.reset_index(drop=True)
df.isna().sum()

#Check for unique values in each column and ensure they're all the same

for column in df.select_dtypes(include="object"):
    print(f"\nColumn: {column}")
    print(df[column].unique())

#Ensure countries and regions follow the UN Region/Country Classification
#In this case Middle East is in the Asia Region and the Sub-region 'South Asia' is updated to its general region Asia
df["primary_country"] = df["primary_country"].replace({"DRC": "Democratic Republic of the Congo", "Russia":"Russian Federation", "Palestine (Gaza)" : "State of Palestine (Gaza)", "Iran" : "Iran (Islamic Republic of)", "Syria" : "Syrian Arab Republic"})

df["region"] = df["region"].replace({"Middle East": "Asia", "South Asia":"Asia"})


Column: conflict_name
['Israel-Hamas War' 'Syrian Civil War' 'Afghanistan War'
 'Russia-Ukraine War' 'DRC Conflict' 'Yemen Civil War' 'Israel-Iran War'
 'Tigray Conflict' 'Iraq War']

Column: conflict_type
['Asymmetric War' 'Civil War' 'Interstate/Counter-insurgency'
 'Interstate War']

Column: region
['Middle East' 'South Asia' 'Europe' 'Africa']

Column: primary_country
['Palestine (Gaza)' 'Syria' 'Afghanistan' 'Ukraine' 'Russia' 'DRC' 'Yemen'
 'Israel' 'Ethiopia' 'Iraq' 'Iran']


In [74]:
#check datatypes are correct 

df.dtypes

conflict_name             object
conflict_type             object
region                    object
start_year                 int64
end_year                   int64
primary_country           object
food_insecurity_rate_    float64
dtype: object

In [75]:
#Look at the statistics to understand if there are any outliers or extreme values 

df.describe()

,start_year,end_year,food_insecurity_rate_
count,33513.000000,33513.000000,33513.000000
mean,2014.732552,2023.973592,23.343890
std,9.655955,4.290853,12.514673
min,1996.000000,2011.000000,3.100000
25%,2003.000000,2022.000000,13.920000
50%,2020.000000,2026.000000,21.500000
75%,2023.000000,2026.000000,30.090000
max,2025.000000,2026.000000,86.320000


In [76]:
df.groupby("conflict_name").size().sort_values(ascending=False)

#Lets check if the number of rows per conflict_name --> there are too many rows per conflict, which means the dataset has many simulated economic scenarios per conflict
#So we need to check each conflict by country because a country can be involved in more than one conflict -- e.g. Israel is involved in two conflicts 

df.groupby("conflict_name")["primary_country"].unique()


df.groupby(["conflict_name","primary_country"]).size()

#We should filter per country per conflict to ensure unique results rather than repeated conflict-country pairs with different scenarios 
df_clean = (df.groupby(["conflict_name","primary_country"]).agg({
        "region":"first",
        "conflict_type":"first",
        "start_year":"first",
        "end_year":"first",
        "food_insecurity_rate_":"mean"
    })
    .reset_index()
)

df_clean

df_clean.sort_values(by="start_year")

,conflict_name,primary_country,region,conflict_type,start_year,end_year,food_insecurity_rate_
1,DRC Conflict,Democratic Republic of the Congo,Africa,Civil War,1996,2026,19.742965
0,Afghanistan War,Afghanistan,Asia,Interstate/Counter-insurgency,2001,2021,19.611456
2,Iraq War,Iraq,Asia,Interstate War,2003,2011,19.808248
9,Syrian Civil War,Syrian Arab Republic,Asia,Civil War,2011,2026,29.812686
11,Yemen Civil War,Yemen,Asia,Civil War,2015,2026,30.218582
10,Tigray Conflict,Ethiopia,Africa,Civil War,2020,2022,19.538480
7,Russia-Ukraine War,Russian Federation,Europe,Interstate War,2022,2026,18.874616
8,Russia-Ukraine War,Ukraine,Europe,Interstate War,2022,2026,18.778144
3,Israel-Hamas War,Israel,Asia,Asymmetric War,2023,2026,28.505579
4,Israel-Hamas War,State of Palestine (Gaza),Asia,Asymmetric War,2023,2026,28.601170


In [77]:
df_clean = df_clean.rename(columns={"food_insecurity_rate_": "food_insecurity_rate"})
df_clean["conflict_type"] = df_clean["conflict_type"].replace({"Interstate/Counter-insurgency" : "Interstate War"})
df_clean = df_clean.rename(columns={"primary_country": "country"})

df_clean.to_csv("conflict_clean.csv", index=False)

df_clean

,conflict_name,country,region,conflict_type,start_year,end_year,food_insecurity_rate
0,Afghanistan War,Afghanistan,Asia,Interstate War,2001,2021,19.611456
1,DRC Conflict,Democratic Republic of the Congo,Africa,Civil War,1996,2026,19.742965
2,Iraq War,Iraq,Asia,Interstate War,2003,2011,19.808248
3,Israel-Hamas War,Israel,Asia,Asymmetric War,2023,2026,28.505579
4,Israel-Hamas War,State of Palestine (Gaza),Asia,Asymmetric War,2023,2026,28.601170
5,Israel-Iran War,Iran (Islamic Republic of),Asia,Interstate War,2025,2026,18.759795
6,Israel-Iran War,Israel,Asia,Interstate War,2025,2026,18.986591
7,Russia-Ukraine War,Russian Federation,Europe,Interstate War,2022,2026,18.874616
8,Russia-Ukraine War,Ukraine,Europe,Interstate War,2022,2026,18.778144
9,Syrian Civil War,Syrian Arab Republic,Asia,Civil War,2011,2026,29.812686


In [78]:
df_clean

,conflict_name,country,region,conflict_type,start_year,end_year,food_insecurity_rate
0,Afghanistan War,Afghanistan,Asia,Interstate War,2001,2021,19.611456
1,DRC Conflict,Democratic Republic of the Congo,Africa,Civil War,1996,2026,19.742965
2,Iraq War,Iraq,Asia,Interstate War,2003,2011,19.808248
3,Israel-Hamas War,Israel,Asia,Asymmetric War,2023,2026,28.505579
4,Israel-Hamas War,State of Palestine (Gaza),Asia,Asymmetric War,2023,2026,28.601170
5,Israel-Iran War,Iran (Islamic Republic of),Asia,Interstate War,2025,2026,18.759795
6,Israel-Iran War,Israel,Asia,Interstate War,2025,2026,18.986591
7,Russia-Ukraine War,Russian Federation,Europe,Interstate War,2022,2026,18.874616
8,Russia-Ukraine War,Ukraine,Europe,Interstate War,2022,2026,18.778144
9,Syrian Civil War,Syrian Arab Republic,Asia,Civil War,2011,2026,29.812686
